# IntentOps — BANKING77 data exploration

Phase 1: verified local data and deterministic preparation, **no trained classifier**.
No cell downloads data or installs packages. Install the optional notebook extras
before launching. Run `python -m ml.dataset fetch` explicitly (or manually copy
original CSVs), then `verify` and `prepare` from the project root.

Original code: MIT. BANKING77: PolyAI, CC BY 4.0; see THIRD_PARTY_NOTICES.md.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from ml.dataset import DEFAULT_RAW, DataError, describe, overlap, split_rows, verify

raw_dir = Path(os.environ.get("INTENTOPS_RAW_DIR", str(DEFAULT_RAW)))
# DEFAULT_RAW is module-relative, so changing notebook working directory is safe.
try:
    datasets, manifest = verify(raw_dir)
except (DataError, OSError) as exc:
    datasets = None
    print(f"Local data not ready: {exc}")
    print("From the repository root: python -m ml.dataset fetch; python -m ml.dataset verify")
    print("Or copy the official train.csv and test.csv into data/raw/ and run verify offline.")

## Identity, schema, and official split

The original files contain `text` and string `category`. Hugging Face's numeric
`label` interface is a different representation. Spelling, punctuation, case,
whitespace and source rows are preserved. The verifier rejects invalid/nonempty
fields, wrong hashes, unexpected counts and category inventories before EDA.
The 3,080 official test rows are reserved for later final assessment.


In [ ]:
if datasets is not None:
    print("Pinned source revision:", manifest["source_revision"])
    display(
        pd.DataFrame(
            [
                {
                    "split": name,
                    "rows": info["rows"],
                    "categories": info["category_count"],
                    "null_or_blank_fields": info["null_or_blank_fields"],
                    "sha256": info["sha256"],
                }
                for name, info in manifest["files"].items()
            ]
        ).set_index("split")
    )
    print("Schema:", manifest["files"]["train.csv"]["schema"])

## Class frequencies and text lengths

These are descriptive data statistics, not classifier metrics. Lengths count
Unicode characters. No full source messages are displayed or exported here.


In [ ]:
if datasets is not None:
    frequencies = pd.DataFrame(
        {name: info["category_counts"] for name, info in manifest["files"].items()}
    ).sort_index()
    display(frequencies)
    display(
        pd.DataFrame(
            {name: info["text_length_characters"] for name, info in manifest["files"].items()}
        )
    )
    display(
        pd.DataFrame(
            [
                {
                    "split": name,
                    "smallest_class": info["class_size_min"],
                    "largest_class": info["class_size_max"],
                    "imbalance_ratio": info["class_imbalance_ratio"],
                    "duplicate_text_excess_rows": info["duplicate_text_excess_rows"],
                    "duplicate_record_excess_rows": info["duplicate_record_excess_rows"],
                    "conflicting_label_text_count": info["conflicting_label_text_count"],
                }
                for name, info in manifest["files"].items()
            ]
        )
    )

## Training/validation membership and leakage checks

Only official training rows enter the stratified 80/20 split (seed 42). IDs are
zero-based data-record indices from original train.csv. Every row is retained,
including duplicates; exact-text overlap is explicitly reported. This policy
preserves the benchmark but does not guarantee duplicate-free or semantic
leakage-free evaluation. Do not train or tune using the reserved official test.
Run `python -m ml.dataset prepare` to save CSVs and a deterministic manifest.


In [ ]:
if datasets is not None:
    original_train = datasets["train.csv"]
    train_ids, validation_ids = split_rows(original_train, seed=42, val_fraction=0.20)
    assert not set(train_ids) & set(validation_ids)
    assert set(train_ids) | set(validation_ids) == set(range(len(original_train)))
    partitions = {
        "train": [original_train[i] for i in train_ids],
        "validation": [original_train[i] for i in validation_ids],
    }
    print("Partition sizes:", {name: len(rows) for name, rows in partitions.items()})
    display(
        pd.DataFrame({name: describe(rows)["category_counts"] for name, rows in partitions.items()})
    )
    print(
        "Train/validation identical-text count:",
        overlap(partitions["train"], partitions["validation"])["identical_text_count"],
    )
    print(
        "Official train/test identical-text count:",
        manifest["official_train_test_overlap"]["identical_text_count"],
    )

## Limitations and provenance

BANKING77 covers English banking support and single-intent labels; other domains,
languages, multi-intent queries and real-world deployment quality are untested.
Exact-match duplicate checks do not detect paraphrases or near-duplicates. No
normalization, deduplication, training, calibration, accuracy, or F1 is performed.

Source: https://github.com/PolyAI-LDN/task-specific-datasets
Metadata: https://huggingface.co/datasets/PolyAI/banking77
Citation: Casanueva et al. (2020), *Efficient Intent Detection with Dual Sentence Encoders*,
https://arxiv.org/abs/2003.04807. Data retains CC BY 4.0 attribution obligations.
Notebook outputs are intentionally cleared before committing.
